# TR48A MT Pipeline
MT post-editing pipeline for Phrase TMS (.mxliff) files using the Gemini API.
Inputs: termbase (.xlsx), MXLIFF file
Output: translated MXLIFF with target segments injected

In [ ]:
# ==========================================
# 0. INSTALLATIONS & IMPORTS
# ==========================================
!pip install -q google-genai tqdm pandas openpyxl

import xml.etree.ElementTree as ET
import pandas as pd
import json
import time
import sys
import getpass
from google import genai
from google.genai import types
from google.colab import files
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION & STYLE GUIDE
# ==========================================
# Insert your initial API Key
API_KEY = "YOUR API KEY HERE"
client = genai.Client(api_key=API_KEY)

# Your strict, LLM-optimized Style Guide
CONDENSED_STYLE_GUIDE = """
1. TONE & ADDRESS: Maintain a warm, respectful, and inclusive tone. Address users with polite formal "Siz". Use "Biz" when the application refers to itself. Use gender-neutral language ("O" or omit pronouns entirely).
2. COURTESY WORDS: Use "lütfen" when the user is inconvenienced or asked to wait. Use "özür dileriz", "maalesef", or "ne yazık ki" ONLY for serious problems/data loss. STRICTLY AVOID the literal translation "üzgünüz".
3. VERBS & VOICE: Call-to-action buttons must use the imperative mood without infinitive suffixes (e.g., Use "Gönder", NOT "Göndermek"). Use present continuous (-yor) for background processes. Use non-blaming phrases for errors (e.g., Use "Şifre hatalı", NOT "Yanlış şifre girdiniz"). Avoid passive structures using "tarafından".
4. NOUNS & PLURALS: Nouns must remain singular when quantified by a number or placeholder (e.g., "%d dosya silindi", NOT "%d dosyalar silindi"). Use undefined noun phrases for UI labels (e.g., "Dosya Özellikleri", NOT "Dosyanın Özellikleri").
5. CAPITALIZATION: Use sentence-case for subheadings, options, and explanatory texts. Title case is acceptable for short UI labels and buttons (1-3 words). Do not capitalize file extensions.
6. PUNCTUATION: Standalone buttons and menu items must NEVER end with a period. Full-sentence error/confirmation messages MUST end with a period. Colons in form fields must touch the preceding word, followed by a space. Use an ellipsis (...) for elements that open a new window or modal.
7. ABBREVIATIONS: Write acronyms without periods (e.g., TDK, ABD), except for "T.C.". Attach case suffixes with an apostrophe based on the pronunciation of the acronym's last letter (e.g., "TDK'nin"). Capitalize data storage units (KB, MB); use lowercase for international measurement units (m, kg).
8. NUMBERS, DATES & SYMBOLS: Use a comma for decimals and a period for thousands (e.g., 1.500,25). Place the percent sign immediately to the left of the number WITHOUT a space (e.g., %100). Insert a space between a number and its unit. Use 24-hour format (HH:mm) and DD.MM.YYYY format for dates.
"""

# ==========================================
# 2. FILE UPLOADS
# ==========================================
print("==========================================")
print("📥 STEP 1: Upload your Termbase (Excel .xlsx)")
print("==========================================")
try:
    tb_uploaded = files.upload()
    tb_file_name = list(tb_uploaded.keys())[0]
    tb_df = pd.read_excel(tb_file_name).fillna("")
    termbase_dict = dict(zip(tb_df.iloc[:, 0].astype(str), tb_df.iloc[:, 1].astype(str)))
    print(f"\n✅ Successfully loaded {len(termbase_dict)} terms from {tb_file_name}!\n")
except Exception as e:
    print(f"\n❌ [ERROR] Failed to load Termbase. Details: {e}")
    sys.exit()

print("==========================================")
print("📥 STEP 2: Upload your Phrase TMS File (.mxliff)")
print("==========================================")
try:
    mxliff_uploaded = files.upload()
    mxliff_file = list(mxliff_uploaded.keys())[0]
    print(f"\n✅ Successfully loaded {mxliff_file}!\n")
except Exception as e:
    print(f"\n❌ [ERROR] Failed to load MXLIFF. Details: {e}")
    sys.exit()

# ==========================================
# 3. HELPER FUNCTIONS & ERROR HANDLING
# ==========================================
def get_relevant_terms(chunk_text, termbase):
    relevant_terms = {}
    chunk_lower = str(chunk_text).lower()
    for src, tgt in termbase.items():
        if src.strip() != "" and src.lower() in chunk_lower:
            relevant_terms[src] = tgt
    return relevant_terms

def process_chunk(chunk, retries=5):
    """Sends a batch of segments. If daily limit is hit, prompts for a new key."""
    global client # Allows us to update the API key globally if needed

    all_text_in_chunk = " ".join(chunk['Source_Text'].astype(str).tolist())
    active_terms = get_relevant_terms(all_text_in_chunk, termbase_dict)

    term_instructions = "Use the following glossary for this batch:\n"
    if active_terms:
        for src, tgt in active_terms.items():
            term_instructions += f"- {src} -> {tgt}\n"
    else:
        term_instructions = "No specific glossary terms for this batch."

    DYNAMIC_PROMPT = f"""You are a localization expert translating software to Turkish.

    STYLE GUIDE:
    {CONDENSED_STYLE_GUIDE}

    {term_instructions}

    Respond ONLY with a valid JSON array of objects. Each object must contain:
    - "id": The exact ID provided
    - "result": The Turkish translation
    """

    prompt_data = chunk[['id', 'Source_Text']].to_json(orient="records")
    full_payload = DYNAMIC_PROMPT + "\n\nDATA TO TRANSLATE:\n" + prompt_data

    # MASTER LOOP: Keeps trying until success or manual skip
    while True:
        for attempt in range(retries):
            try:
                response = client.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=full_payload,
                    config=types.GenerateContentConfig(
                        temperature=0.0,
                        response_mime_type="application/json"
                    )
                )
                return json.loads(response.text)

            except Exception as e:
                error_msg = str(e)

                if "429" in error_msg or "Quota" in error_msg:
                    print(f"\n⏳ [API LIMIT] Waiting 60s for Google's API to refresh... (Attempt {attempt+1} of {retries})")
                    time.sleep(60)

                elif "400" in error_msg or "Bad Request" in error_msg:
                    print(f"\n⚠️ [BAD REQUEST] The payload might be too large. Retrying... (Attempt {attempt+1} of {retries})")
                    time.sleep(5)

                elif "500" in error_msg or "503" in error_msg:
                    print(f"\n⚠️ [SERVER ERROR] Google's servers are busy. Retrying... (Attempt {attempt+1} of {retries})")
                    time.sleep(10)

                else:
                    print(f"\n❌ [UNKNOWN ERROR] {error_msg}. Retrying... (Attempt {attempt+1} of {retries})")
                    time.sleep(5)

        # If the code reaches here, it means all 5 retries failed (Daily Limit Hit)
        print("\n🚨 [DAILY LIMIT REACHED] The script failed 5 times in a row.")
        print("This means your current API key has run out of its Daily Quota.")
        print("Please generate a new API key using a different Google account.")

        # Pause script and open an input box
        new_key = getpass.getpass("🔑 Paste NEW API Key here (or type 'skip' to skip this chunk): ").strip()

        if new_key.lower() == 'skip':
            print("⏭️ Skipping chunk as requested...")
            return None

        if new_key:
            # Swap out the API engine with the new key
            client = genai.Client(api_key=new_key)
            print("✅ New API Key accepted! Resuming exactly where we left off...\n")
            # The 'while True' loop will now restart the attempt cycle from 0 with the new key!
        else:
            print("❌ Invalid input. Try again.")

# ==========================================
# 4. MXLIFF PARSING & EXTRACTION
# ==========================================
ET.register_namespace('', "urn:oasis:names:tc:xliff:document:1.2")
ET.register_namespace('m', "http://www.memsource.com/mxlf/2.0")
ns = {'xliff': 'urn:oasis:names:tc:xliff:document:1.2'}

print("==========================================")
print("⚙️ EXTRACTING SEGMENTS FROM MXLIFF...")
print("==========================================")
tree = ET.parse(mxliff_file)
root = tree.getroot()

segments = []
for tu in root.findall('.//xliff:trans-unit', ns):
    tu_id = tu.get('id')
    source_node = tu.find('xliff:source', ns)

    if source_node is not None and source_node.text and str(source_node.text).strip() != "":
        segments.append({'id': tu_id, 'Source_Text': str(source_node.text)})

df = pd.DataFrame(segments)

if df.empty:
    print("\n❌ [ERROR] No translatable segments found in the MXLIFF file.")
    sys.exit()

print(f"✅ Found {len(df)} translatable segments.\n")

# ==========================================
# 5. CHUNKING & TRANSLATION
# ==========================================
CHUNK_SIZE = 30
all_results = {}
chunks = [df[i:i + CHUNK_SIZE] for i in range(0, df.shape[0], CHUNK_SIZE)]

print("==========================================")
print("🚀 SENDING DATA TO GEMINI API...")
print("==========================================")

for chunk in tqdm(chunks, desc="Translating Chunks"):
    chunk_results = process_chunk(chunk)

    if chunk_results:
        for item in chunk_results:
            if isinstance(item, dict) and 'id' in item and 'result' in item:
                all_results[item['id']] = item['result']

    time.sleep(2)

# ==========================================
# 6. INJECT TRANSLATIONS & EXPORT MXLIFF
# ==========================================
print("\n==========================================")
print("💾 SAVING TRANSLATIONS BACK TO MXLIFF...")
print("==========================================")

for tu in root.findall('.//xliff:trans-unit', ns):
    tu_id = tu.get('id')

    if tu_id in all_results:
        target_node = tu.find('xliff:target', ns)
        if target_node is None:
            target_node = ET.SubElement(tu, '{urn:oasis:names:tc:xliff:document:1.2}target')

        target_node.text = str(all_results[tu_id])

output_name = f"translated_{mxliff_file}"
tree.write(output_name, encoding="utf-8", xml_declaration=True)

print(f"\n🎉 PROCESS COMPLETE! Downloading: {output_name}")
files.download(output_name)